# Training an anomaly detector without a single defect

This notebook fits a defect detector on photographs of **correct** circuit boards and nothing else.
No defect images, no masks, no annotation of any kind.

The idea is a bet about manufacturing: good units are the output of the process and defective ones
are the exception it is designed to avoid, so images of correct parts are plentiful and images of
faults are not. Instead of learning what is wrong, the model learns in fine detail what *correct*
looks like and flags whatever it cannot reconcile with that. The failure mode it cannot have is
"I was not trained on that kind of defect".

**What you will do here**

1. Download one VisA circuit-board product
2. Confirm for yourself that the training split contains no defects
3. Meet the anomalib evaluation defaults that will quietly ruin your numbers, and see the proof
4. Look inside the model, printing the real tensor shapes rather than trusting a diagram
5. Train on good boards only
6. Score the held-out split

**What this notebook is not.** It trains **one** product, `pcb3`, so it does not reproduce the
article's headline table. Those figures come from all four products trained as a single model over a
merged directory of symlinks, and this repository does not build that directory. What you get here
is the same method at the same per-image training budget, on one board. Expect image-AUROC near
0.99. AUPIMO is the volatile one: our runs of this model on this board have read anywhere between
0.69 and 0.76, so do not treat a single figure from it as precise.

**What you need.** A CUDA GPU: training and the test pass together peaked at 5.5 GB reserved on our
run, so 8 GB is comfortable. Also about 3.5 GB of disk, being 1.8 GB of Hugging Face parquet cache plus 1.4 GB of
decoded `pcb3` images.

**Runtime.** About 17 minutes on an RTX 5090, roughly 15 of them the 5,000 training steps and 90
seconds the final test pass. Other hardware was not timed. The four-product checkpoint used by
`02_inspect.ipynb` took 38 minutes on the same card at 20,000 steps.
If you only want to see the detector work, skip to `02_inspect.ipynb`, which downloads the
four-product checkpoint instead of training one.

In [ ]:
import sys, subprocess
from pathlib import Path

HERE = Path.cwd()
REPO = next((p for p in [HERE, *HERE.parents] if (p / "pcb_anomaly").is_dir()), None)
if REPO is None:
    raise SystemExit(f"no repository root above {HERE}. Start Jupyter inside the clone.")
sys.path.insert(0, str(REPO))

import torch
print("torch     ", torch.__version__)
print("cuda       ", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
import anomalib
print("anomalib  ", anomalib.__version__)
print("repo      ", REPO)

## 1. The data

VisA is a public industrial inspection benchmark. We use one of its four printed circuit board
products. `pcb3` is a good starting point: an infrared sensor board, and its defects sit in the
middle of the difficulty range.

The download script writes an anomalib-style folder layout. It fetches the whole VisA parquet set
regardless of which category you ask for, so the first run takes a while.

In [ ]:
CATEGORY = "pcb3"
DATA = REPO / "data/visa" / CATEGORY

if not sorted((DATA / "train/good").glob("*.png")):
    # Jupyter does not show a child process's stdout, so this is captured and reprinted when the
    # download finishes. Nothing appears in between. That is not a hang.
    print(f"downloading VisA {CATEGORY}. This pulls the whole parquet set, several GB, so the "
          f"first run takes a while and prints nothing until it is done.", flush=True)
    r = subprocess.run([sys.executable, str(REPO / "scripts/get_visa.py"),
                        "--categories", CATEGORY], capture_output=True, text=True)
    print(r.stdout[-2000:] or "(no output)")
    if r.returncode:
        raise SystemExit(f"get_visa.py failed:\n{r.stderr[-2000:]}")

for split in ["train/good", "test/good", "test/bad", "ground_truth/bad"]:
    n = len(list((DATA / split).glob("*"))) if (DATA / split).exists() else 0
    print(f"  {split:20s} {n:4d}")

### The premise, checked rather than asserted

The whole method rests on the training split containing no defects. That is worth verifying rather
than believing, because if a single defective board leaks in, the model learns it as normal and will
never flag it again.

In [ ]:
train_files = sorted((DATA / "train/good").glob("*.png"))
masks       = sorted((DATA / "ground_truth/bad").glob("*.png"))
defective_in_train = sorted((DATA / "train").glob("**/bad/*")) if (DATA / "train").exists() else []

print(f"training images        {len(train_files)}")
print(f"of which defective     {len(defective_in_train)}")
print(f"masks available        {len(masks)}  (test split only, scored against, never fitted on)")
assert len(defective_in_train) == 0, "a defective image reached the training split"

In [ ]:
import matplotlib.pyplot as plt
import cv2

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, f in zip(axes, train_files[:4]):
    ax.imshow(cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB))
    ax.set_title(f.stem, fontsize=8); ax.axis("off")
fig.suptitle("Four of the boards the model will learn from. Every one of them is correct.", y=1.02)
plt.tight_layout(); plt.show()

## 2. Two ways anomalib will quietly ruin your evaluation

Neither raises an error. Both cost us real time.

**The split.** `val_split_mode` defaults to `FROM_TEST` with `val_split_ratio=0.5`, so half your test
set is consumed as a validation split. Your metrics are then computed on roughly half the images you
believe you are evaluating on, and the draw is random per run. We saw AUPIMO readings move by more
than 0.4 between runs of a configuration we had not changed.

**The seed.** `seed` is unset by default, which is at least visible. The trap is that setting it to
zero does not fix it either: anomalib checks the seed for truthiness, and zero is falsy, so `seed=0`
silently means unseeded. And `Folder(seed=...)` pins only the data split, so the setup below calls
Lightning's `seed_everything` as well to pin the rest of the run.

Rather than take either on trust, read them out of the installed library.

In [ ]:
import inspect as _inspect
import anomalib.data.utils.split as _split
from anomalib.data import Folder
from anomalib.data.utils import ValSplitMode

sig = _inspect.signature(Folder.__init__)
for k in ["val_split_mode", "val_split_ratio", "seed"]:
    print(f"  {k:18s} default = {sig.parameters[k].default!r}")

hits = [l.strip() for l in Path(_split.__file__).read_text().splitlines() if "manual_seed" in l]
print("\n  " + (hits[0] if hits else f"no manual_seed line in {_split.__file__}"))
print("  ^ `if seed` is falsy for 0, so seed=0 silently disables seeding")

### The corrected setup

`FROM_TRAIN` with a small ratio takes validation from the good training images instead. The full
test set is then tested, there is no leakage, and the normalisation and threshold statistics are
fitted on good images only, which is what a deployment actually has.

One consequence to expect: because the validation split is now all-normal, anomalib warns once per
epoch that its adaptive threshold will be the highest normal score it saw. That warning is correct
and harmless here. It concerns anomalib's own pass/fail threshold, which nothing downstream uses:
AUROC and AUPIMO are threshold-free, and the accept/reject limit in `02_inspect.ipynb` is calibrated
separately on good boards.

In [ ]:
from lightning.pytorch import seed_everything

seed_everything(1, workers=True)   # Folder's own seed covers the split and nothing else

datamodule = Folder(
    name=f"visa_{CATEGORY}",
    root=DATA,
    normal_dir="train/good",         # the only images used for fitting
    abnormal_dir="test/bad",         # evaluation only, never seen during training
    normal_test_dir="test/good",
    mask_dir="ground_truth/bad",
    train_batch_size=8,
    eval_batch_size=8,

    # The two overrides that matter.
    val_split_mode=ValSplitMode.FROM_TRAIN,
    val_split_ratio=0.05,
    seed=1,
)
datamodule.setup()
print(f"  train {len(datamodule.train_data):4d}   (good only)")
print(f"  val   {len(datamodule.val_data):4d}   (good only, drawn from train)")
print(f"  test  {len(datamodule.test_data):4d}   (the full held-out split, nothing consumed)")

## 3. Inside the model

Dinomaly's subtitle is its thesis: *The Less Is More Philosophy in Multi-Class Unsupervised Anomaly
Detection*. The mechanism is reconstruction. A frozen encoder describes the image, a small decoder is
trained to reproduce those descriptions, and because the decoder only ever practised on good boards,
whatever it cannot rebuild was never part of what "good" looks like. **The reconstruction error is
the anomaly map.**

The design then goes out of its way to stop the decoder becoming too capable, because a decoder that
can rebuild anything will happily rebuild a defect and report nothing wrong. Four choices do that:
a dropout bottleneck, linear attention that focuses poorly on purpose, a loss that down-weights
points already reconstructed well, and comparison in fused layer groups rather than layer by layer.

Rather than describe the architecture, print it.

In [ ]:
from anomalib.metrics import AUPIMO, AUROC, Evaluator
from anomalib.models import Dinomaly

# Dinomaly's default test metrics are image and pixel AUROC and F1. AUPIMO is the one worth ranking
# on, so it has to be asked for. strict=False lets it return NaN instead of raising when a map is
# degenerate, which happens if you cut the step budget down to a smoke test.
model = Dinomaly(evaluator=Evaluator(test_metrics=[
    AUROC(fields=["pred_score", "gt_label"], prefix="image_"),
    AUROC(fields=["anomaly_map", "gt_mask"], prefix="pixel_"),
    AUPIMO(fields=["anomaly_map", "gt_mask"], strict=False),
]))

enc = model.model.encoder
vit = enc.feature_extractor      # the ViT itself; enc is a TimmFeatureExtractor wrapper

frozen    = sum(p.numel() for p in enc.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"  frozen encoder      {frozen:>12,}   DINOv2 ViT-B/14, not one weight changes")
print(f"  trained             {trainable:>12,}   bottleneck + decoder, fitted on good boards only")
print()
print(f"  width               {vit.embed_dim}  = {vit.blocks[0].attn.num_heads} heads x "
      f"{vit.embed_dim // vit.blocks[0].attn.num_heads} dims per head")
print(f"  patch size          {vit.patch_embed.patch_size}")
print(f"  patch projection    {tuple(vit.patch_embed.proj.weight.shape)}"
      f"  = {vit.patch_embed.proj.weight.numel():,} numbers")

A patch *grows* on its way in rather than shrinking. Its `3 x 14 x 14 = 588` raw pixel values are
projected up to 768 features, and the width stays 768 through every block.

The token count is pure geometry. Anomalib resizes the input to 448 pixels and centre-crops it to
392, and `392 / 14 = 28` across and down, so `28 x 28 = 784` patch tokens. Five more are prepended,
one CLS and four registers. Confirm it with a real forward pass.

In [ ]:
seen = {}
h = model.model.encoder.feature_extractor.blocks[0].register_forward_hook(
    lambda m, i, o: seen.update(shape=tuple(i[0].shape)))
with torch.no_grad():
    model.model.encoder(torch.zeros(1, 3, 392, 392))
h.remove()

b, tokens, width = seen["shape"]
print(f"  sequence entering block 0:  {tokens} x {width}")
print(f"    {tokens - 5} patch tokens  (28 x 28)")
print(f"    1 CLS + 4 register tokens")

In [ ]:
print("  tapped encoder layers :", model.model.target_layers)
print("  fusion groups         :", model.model.fuse_layer_encoder)
print("  decoder depth         :", len(model.model.bottleneck), "bottleneck /",
      len(model.model.decoder), "decoder blocks")
print("  loss                  :", type(model.model.loss_fn).__name__)

Blocks 0 and 1 are still edge and texture detectors, and the last two have specialised toward
DINOv2's own self-supervised objective. The middle is where general, transferable structure lives,
which is why the extractor hands back `blocks.2` through `blocks.9`.

## 4. Train

Only the bottleneck and the decoder are fitted. The encoder never moves.

Anomalib's default budget is 5,000 steps, which is written for a single product and is what we use
here. If you train a unified model across several products the dataset is several times larger and
the same step count gives each image a fraction of its intended exposure, so scale the budget to
keep epochs-per-image constant. Getting this wrong is not a small effect: at 5,000 steps our
four-product model scored mean AUPIMO 0.349 against 0.519 for four separate ones, and at 20,000
steps, which restores the same epochs-per-image, the same configuration reached 0.589 and beat them.

In [ ]:
from anomalib.engine import Engine

MAX_STEPS = 5_000       # one product. Four products as one model need 20_000 for the same exposure.

engine = Engine(max_steps=MAX_STEPS, accelerator="gpu", devices=1)
engine.fit(model=model, datamodule=datamodule)

## 5. Score the held-out split

`AUROC` answers "given one good board and one defective board, how often is the defective one scored
higher". `AUPIMO` is the more informative of the two and is the metric we asked the evaluator for
above: pixel AUROC is weak here because over 99 percent of pixels are ordinary background, so a model
scores well simply by ranking the board area low. AUPIMO measures overlap **per image**, at a
threshold where almost nothing on a good image fires, so one badly localised board cannot hide behind
easy background pixels from other boards. It is scored per defective image; the evaluator reports
the mean.

In [ ]:
import numpy as np

results = engine.test(model=model, datamodule=datamodule)
for k, v in results[0].items():
    a = np.asarray(torch.as_tensor(v).cpu() if torch.is_tensor(v) else v, dtype=float).ravel()
    note = f"   (mean over {a.size} defective images)" if a.size > 1 else ""
    print(f"  {k:28s} {np.nanmean(a):.4f}{note}")

## 6. Save the weights

Only the trained parts are saved. The frozen DINOv2 encoder is not in the file: timm downloads it
whenever the model is constructed, so shipping a copy would add about 330 MB for nothing.

In [ ]:
ART = REPO / "artifacts"; ART.mkdir(exist_ok=True)
# Deliberately NOT artifacts/detector.pt. That name belongs to the four-product checkpoint that 02
# downloads, and overwriting it would destroy a 234 MB file you would have to fetch again. It would
# also leave your weights paired with thresholds.json, whose limits were calibrated against that
# checkpoint, not yours.
out = ART / f"detector_{CATEGORY}_{MAX_STEPS // 1000}k.pt"
trained = {k: v for k, v in model.state_dict().items() if not k.startswith("model.encoder.")}
torch.save(trained, out)
print(f"wrote {out.relative_to(REPO)}  ({out.stat().st_size / 1048576:.0f} MB)")
print("\nartifacts/thresholds.json holds limits for the four-product checkpoint, not for this one.")
print("To use these weights for accept/reject you would recalibrate on your own good boards.")
print("\nNext: 02_inspect.ipynb runs the four-product checkpoint end to end.")